# 🏆 Module 3.5: Policies and Optimality

## Finding the Best Strategy for Sequential Decision Making

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_03_RL_Mathematical_Foundations/05_Policy_And_Optimality/policy_and_optimality.ipynb)

---

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand deterministic vs stochastic policies** — how agents represent their decision-making strategies
2. **Master policy evaluation and policy improvement** — the two fundamental building blocks
3. **Implement a complete policy iteration algorithm** — alternating evaluation and improvement until optimality
4. **Implement value iteration algorithm** — a more efficient single-pass alternative
5. **Prove and understand convergence guarantees** — why these algorithms always find the optimal solution
6. **Apply optimal policies to image enhancement MDPs** — bridging RL theory with practical image processing

---

In [ ]:
!pip install numpy matplotlib scipy pillow -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
np.random.seed(42)
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset for RL demonstrations
import torchvision
import numpy as np

# MNIST as discrete image states for MDP demonstrations
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
# Use small subset as "states" in our MDP
sample_states = [np.array(mnist[i][0]) for i in range(100)]
print(f"✅ MNIST loaded: Using {len(sample_states)} real images as MDP states")
print("   Each state is a 28x28 grayscale image = 784-dimensional state vector")

## Deep Derivation: Policy Optimality Theorems

### Step 1: Partial Ordering on Policies
**Definition:** $\pi \geq \pi'$ iff $V^\pi(s) \geq V^{\pi'}(s)$ for ALL $s \in \mathcal{S}$.

### Step 2: Policy Improvement Theorem (Full Proof)
**Theorem:** If $Q^\pi(s, \pi'(s)) \geq V^\pi(s)$ for all $s$, then $V^{\pi'}(s) \geq V^\pi(s)$ for all $s$.

**Proof:**
$$V^\pi(s) \leq Q^\pi(s, \pi'(s)) = E[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t=s, A_t=\pi'(s)]$$
$$\leq E[R_{t+1} + \gamma Q^\pi(S_{t+1}, \pi'(S_{t+1})) | S_t=s, A_t=\pi'(s)]$$
$$= E[R_{t+1} + \gamma R_{t+2} + \gamma^2 V^\pi(S_{t+2}) | S_t=s, A_t=\pi'(s), A_{t+1}=\pi'(S_{t+1})]$$
$$\leq E[R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \gamma^3 V^\pi(S_{t+3}) | \ldots]$$

Repeating infinitely:
$$V^\pi(s) \leq E_{\pi'}\left[\sum_{k=0}^\infty \gamma^k R_{t+k+1} \mid S_t = s\right] = V^{\pi'}(s) \quad \blacksquare$$

### Step 3: Existence of an Optimal Policy (Proof)
**Theorem:** For any finite MDP, there exists a deterministic optimal policy $\pi^*$ such that $\pi^* \geq \pi$ for all $\pi$.

**Proof:**
Define $\pi^*(s) = \arg\max_a Q^*(s, a)$ where $Q^*(s,a) = \max_\pi Q^\pi(s,a)$.

For any policy $\pi$ and any state $s$:
$$V^{\pi^*}(s) = \max_a Q^*(s,a) \geq \sum_a \pi(a|s) Q^*(s,a) \geq \sum_a \pi(a|s) Q^\pi(s,a) = V^\pi(s)$$

The first inequality holds because $\max \geq$ weighted average. The second holds by definition of $Q^*$. $\blacksquare$

### Step 4: Policy Iteration Convergence (Proof)

**Algorithm:** Repeat: (1) Evaluate $V^\pi$, (2) Improve: $\pi'(s) = \arg\max_a Q^\pi(s, a)$

**Proof of convergence:**
- By the Policy Improvement Theorem: $V^{\pi_{k+1}}(s) \geq V^{\pi_k}(s)$ for all $s$ (monotone improvement)
- Value functions are bounded: $|V^\pi(s)| \leq \frac{R_{\max}}{1-\gamma}$ for all $\pi, s$
- For finite MDPs: there are at most $|\mathcal{A}|^{|\mathcal{S}|}$ deterministic policies
- A bounded monotone sequence over a finite set must converge in finite steps
- At convergence: $V^{\pi_{k+1}} = V^{\pi_k}$, which means $\pi_k$ satisfies the Bellman optimality equation $\blacksquare$

### Step 5: Value Iteration as a Special Case
Value iteration combines evaluation and improvement into one step:
$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V_k(s')]$$

**Proof of convergence rate:**
$$\|V_{k+1} - V^*\|_\infty = \|\mathcal{T}V_k - \mathcal{T}V^*\|_\infty \leq \gamma\|V_k - V^*\|_\infty$$

By induction: $\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$

To achieve $\epsilon$-accuracy: $k \geq \frac{\log(\|V_0 - V^*\|_\infty / \epsilon)}{\log(1/\gamma)}$ iterations $\blacksquare$

### Step 6: Greedy Policy is Sufficient (No Need for Stochastic Optimal Policy)
**Theorem:** There always exists a deterministic optimal policy.

**Proof:** $\pi^*(s) = \arg\max_a Q^*(s,a)$ achieves $V^*(s) = \max_a Q^*(s,a)$.

Any stochastic policy $\pi$ with $\pi(a|s) > 0$ for suboptimal $a$ yields:
$$V^\pi(s) = \sum_a \pi(a|s)Q^*(s,a) < \max_a Q^*(s,a) = V^*(s)$$

unless all actions with $\pi(a|s) > 0$ are optimal. $\blacksquare$

### RL Connection: Optimal Image Enhancement Policy
The optimal policy $\pi^*$ for image enhancement maps each image state to the BEST processing operation:
$$\pi^*(I) = \arg\max_a [R(I, a, f_a(I)) + \gamma V^*(f_a(I))]$$

Policy iteration finds this by repeatedly evaluating "how good is the current strategy?" and improving it — converging to the globally optimal image processing pipeline.

---

## 📍 Section 1: What is a Policy?

A **policy** is the agent's strategy — it tells the agent what action to take in every state. Policies are the central object of study in reinforcement learning because the goal is always to find the *best* policy.

### Deterministic Policies

A **deterministic policy** is a direct mapping from states to actions:

$$\pi: \mathcal{S} \to \mathcal{A}, \quad a = \pi(s)$$

Given a state $s$, the policy outputs exactly one action $a$.

### Stochastic Policies

A **stochastic policy** maps each state to a *probability distribution* over actions:

$$\pi: \mathcal{S} \times \mathcal{A} \to [0, 1], \quad \pi(a|s) = P(A_t = a \mid S_t = s)$$

The key constraint is that the probabilities must sum to one for each state:

$$\sum_{a \in \mathcal{A}} \pi(a|s) = 1 \quad \text{for all } s \in \mathcal{S}$$

### Relationship

Every deterministic policy is a special case of a stochastic policy where all probability mass is on a single action:

$$\pi(a|s) \in \{0, 1\} \quad \text{for all } s, a$$

Stochastic policies are important for:
- **Exploration:** randomly trying different actions to discover better strategies
- **Partially observable environments:** where randomness can be optimal
- **Multi-agent settings:** where predictable behavior can be exploited

In [ ]:
def draw_policy_arrows(ax, policy, title, grid_size=4):
    """Draw arrows representing a policy on a grid world."""
    action_dx = {0: 0, 1: 0, 2: -1, 3: 1}   # up, down, left, right
    action_dy = {0: 1, 1: -1, 2: 0, 3: 0}

    ax.set_xlim(-0.5, grid_size - 0.5)
    ax.set_ylim(-0.5, grid_size - 0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')

    for i in range(grid_size):
        for j in range(grid_size):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        fill=False, edgecolor='gray', linewidth=1))

    for i in range(grid_size):
        for j in range(grid_size):
            state = i * grid_size + j
            if state in [0, grid_size * grid_size - 1]:
                ax.plot(j, grid_size - 1 - i, 's', color='gold', markersize=20, zorder=3)
                ax.text(j, grid_size - 1 - i, 'T', ha='center', va='center',
                        fontsize=10, fontweight='bold', zorder=4)
                continue

            probs = policy[state]
            for a_idx, prob in enumerate(probs):
                if prob > 0.01:
                    dx = action_dx[a_idx] * 0.35 * prob / max(probs)
                    dy = action_dy[a_idx] * 0.35 * prob / max(probs)
                    ax.annotate('', xy=(j + dx, (grid_size - 1 - i) + dy),
                                xytext=(j, grid_size - 1 - i),
                                arrowprops=dict(arrowstyle='->', color='steelblue',
                                                lw=1.5 + 2 * prob, mutation_scale=10 + 10 * prob))

    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(False)

grid_size = 4
n_states = grid_size * grid_size
n_actions = 4

# --- Deterministic policy: always move toward bottom-right terminal ---
det_policy = np.zeros((n_states, n_actions))
for i in range(grid_size):
    for j in range(grid_size):
        s = i * grid_size + j
        if s in [0, n_states - 1]:
            det_policy[s] = [0.25, 0.25, 0.25, 0.25]
            continue
        if i < grid_size - 1 and j < grid_size - 1:
            det_policy[s, 1] = 0.5
            det_policy[s, 3] = 0.5
        elif i < grid_size - 1:
            det_policy[s, 1] = 1.0
        elif j < grid_size - 1:
            det_policy[s, 3] = 1.0
        else:
            det_policy[s, 1] = 1.0

# --- Uniform random policy ---
uniform_policy = np.ones((n_states, n_actions)) / n_actions

# --- Stochastic policy with varying probabilities ---
stoch_policy = np.zeros((n_states, n_actions))
for i in range(grid_size):
    for j in range(grid_size):
        s = i * grid_size + j
        if s in [0, n_states - 1]:
            stoch_policy[s] = [0.25, 0.25, 0.25, 0.25]
            continue
        probs = np.random.dirichlet([1, 1, 1, 1])
        if i < grid_size // 2:
            probs[1] += 0.5
        if j < grid_size // 2:
            probs[3] += 0.5
        stoch_policy[s] = probs / probs.sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

draw_policy_arrows(axes[0], det_policy, '1. Deterministic Policy\n(fixed action per state)')
draw_policy_arrows(axes[1], uniform_policy, '2. Uniform Random Policy\n(equal probability all dirs)')
draw_policy_arrows(axes[2], stoch_policy, '3. Stochastic Policy\n(varying probabilities)')

plt.tight_layout()
plt.savefig('policy_types.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Three policy types visualized on a 4\u00d74 grid world.")
print("   Gold squares marked 'T' are terminal states.")
print("   Arrow thickness/length indicates action probability.")

---

## 📊 Section 2: Policy Evaluation

**Policy evaluation** answers the question: *given a fixed policy $\pi$, how good is each state?*

### Iterative Policy Evaluation Algorithm

**Input:** MDP $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ and policy $\pi$

**Output:** Value function $V^\pi$

**Algorithm:**

1. Initialize $V(s) = 0$ for all $s \in \mathcal{S}$
2. **Repeat:**
   - $\Delta \leftarrow 0$
   - For each $s \in \mathcal{S}$:
     - $v \leftarrow V(s)$
     - $V(s) \leftarrow \displaystyle\sum_a \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V(s') \right]$
     - $\Delta \leftarrow \max(\Delta,\; |v - V(s)|)$
3. **Until** $\Delta < \theta$ (a small threshold)

### Why Does This Converge?

The Bellman expectation operator $T^\pi$ defined by:

$$(T^\pi V)(s) = \sum_a \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V(s') \right]$$

is a **contraction mapping** with factor $\gamma < 1$. By the Banach Fixed Point Theorem, repeated application converges to the unique fixed point $V^\pi$.

In [ ]:
class GridWorldMDP:
    """A simple grid world MDP for demonstrating RL algorithms."""

    def __init__(self, size=5, gamma=0.9):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4  # 0=up, 1=down, 2=left, 3=right
        self.gamma = gamma
        self.terminal_states = [0, self.n_states - 1]

        self.action_deltas = {
            0: (-1, 0),  # up
            1: (1, 0),   # down
            2: (0, -1),  # left
            3: (0, 1),   # right
        }

        self.P = np.zeros((self.n_states, self.n_actions, self.n_states))
        self.R = np.full((self.n_states, self.n_actions), -1.0)
        self._build_transitions()

    def _state_to_rc(self, s):
        return s // self.size, s % self.size

    def _rc_to_state(self, r, c):
        return r * self.size + c

    def _build_transitions(self):
        for s in range(self.n_states):
            if s in self.terminal_states:
                for a in range(self.n_actions):
                    self.P[s, a, s] = 1.0
                    self.R[s, a] = 0.0
                continue
            r, c = self._state_to_rc(s)
            for a in range(self.n_actions):
                dr, dc = self.action_deltas[a]
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.size and 0 <= nc < self.size:
                    s_next = self._rc_to_state(nr, nc)
                else:
                    s_next = s
                self.P[s, a, s_next] = 1.0


def policy_evaluation(mdp, policy, theta=1e-6, max_iter=1000, track_iters=None):
    """Iterative policy evaluation. Returns V and history of (delta, V snapshots)."""
    V = np.zeros(mdp.n_states)
    deltas = []
    V_history = {0: V.copy()}

    for it in range(1, max_iter + 1):
        delta = 0.0
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            v_old = V[s]
            v_new = 0.0
            for a in range(mdp.n_actions):
                v_new += policy[s, a] * (
                    mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a], V)
                )
            V[s] = v_new
            delta = max(delta, abs(v_old - v_new))
        deltas.append(delta)
        if track_iters and it in track_iters:
            V_history[it] = V.copy()
        if delta < theta:
            V_history[it] = V.copy()
            break

    return V, deltas, V_history


mdp = GridWorldMDP(size=5, gamma=0.9)

uniform_pi = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions

track_at = {1, 2, 5, 10}
V_pi, deltas, V_hist = policy_evaluation(mdp, uniform_pi, track_iters=track_at)

final_iter = max(V_hist.keys())
show_iters = sorted([0, 1, 2, 5, 10, final_iter])
show_iters = [k for k in show_iters if k in V_hist]
while len(show_iters) < 6 and show_iters[-1] < final_iter:
    show_iters.append(final_iter)
show_iters = sorted(set(show_iters))[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
vmin, vmax = V_pi.min(), V_pi.max()

for idx, (ax, it) in enumerate(zip(axes.flat, show_iters)):
    V_grid = V_hist[it].reshape(mdp.size, mdp.size)
    im = ax.imshow(V_grid, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    for r in range(mdp.size):
        for c in range(mdp.size):
            ax.text(c, r, f'{V_grid[r, c]:.1f}', ha='center', va='center',
                    fontsize=8, fontweight='bold')
    label = f'Iteration {it}' if it < final_iter else f'Converged (iter {it})'
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes, shrink=0.6, label='V(s)')
fig.suptitle('Policy Evaluation: Value Function Evolution (Uniform Random Policy)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('policy_evaluation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.semilogy(range(1, len(deltas) + 1), deltas, 'o-', color='crimson', markersize=3)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Max |\u0394V|  (log scale)', fontsize=12)
ax2.set_title('Policy Evaluation Convergence', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('policy_evaluation_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n\u2705 Policy evaluation converged in {len(deltas)} iterations.")
print(f"   Final max delta: {deltas[-1]:.2e}")
print(f"   V* range: [{V_pi.min():.2f}, {V_pi.max():.2f}]")

---

## 🚀 Section 3: Policy Improvement

Given the value function $V^\pi$ of a policy $\pi$, can we find a **better** policy?

### Policy Improvement Theorem

**Theorem.** Let $\pi$ and $\pi'$ be two policies. If for all states $s \in \mathcal{S}$:

$$Q^\pi(s, \pi'(s)) \geq V^\pi(s)$$

then $\pi'$ is at least as good as $\pi$:

$$V^{\pi'}(s) \geq V^{\pi}(s) \quad \text{for all } s \in \mathcal{S}$$

### Proof Sketch

Starting from any state $s$:

$$V^\pi(s) \leq Q^\pi(s, \pi'(s)) = \mathbb{E}\left[ R_{t+1} + \gamma V^\pi(S_{t+1}) \mid S_t = s, A_t = \pi'(s) \right]$$

$$\leq \mathbb{E}\left[ R_{t+1} + \gamma Q^\pi(S_{t+1}, \pi'(S_{t+1})) \mid S_t = s, A_t = \pi'(s) \right]$$

Unrolling this recursion all the way yields $V^\pi(s) \leq V^{\pi'}(s)$.

### Greedy Policy Improvement

The most natural way to improve a policy is to act **greedily** with respect to $V^\pi$:

$$\pi'(s) = \arg\max_a Q^\pi(s, a) = \arg\max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s') \right]$$

This always satisfies the policy improvement theorem, so the new policy is guaranteed to be at least as good!

In [ ]:
def compute_q_values(mdp, V):
    """Compute Q(s,a) for all state-action pairs given V."""
    Q = np.zeros((mdp.n_states, mdp.n_actions))
    for s in range(mdp.n_states):
        for a in range(mdp.n_actions):
            Q[s, a] = mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a], V)
    return Q


def greedy_policy(mdp, V):
    """Return the greedy deterministic policy w.r.t. V (as a stochastic array)."""
    Q = compute_q_values(mdp, V)
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    for s in range(mdp.n_states):
        best_actions = np.where(Q[s] == Q[s].max())[0]
        for ba in best_actions:
            policy[s, ba] = 1.0 / len(best_actions)
    return policy


def draw_grid_policy_and_value(ax_pol, ax_val, mdp, policy, V, pol_title, val_title):
    """Draw policy arrows and value heatmap side by side."""
    action_dx = {0: 0, 1: 0, 2: -0.35, 3: 0.35}
    action_dy = {0: 0.35, 1: -0.35, 2: 0, 3: 0}

    sz = mdp.size
    ax_pol.set_xlim(-0.5, sz - 0.5)
    ax_pol.set_ylim(-0.5, sz - 0.5)
    ax_pol.set_aspect('equal')
    ax_pol.set_title(pol_title, fontsize=11, fontweight='bold')

    for i in range(sz):
        for j in range(sz):
            ax_pol.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                           fill=False, edgecolor='gray'))
    for i in range(sz):
        for j in range(sz):
            s = i * sz + j
            y_pos = sz - 1 - i
            if s in mdp.terminal_states:
                ax_pol.plot(j, y_pos, 's', color='gold', markersize=18)
                ax_pol.text(j, y_pos, 'T', ha='center', va='center',
                           fontsize=9, fontweight='bold')
                continue
            for a_idx in range(mdp.n_actions):
                prob = policy[s, a_idx]
                if prob > 0.01:
                    dx = action_dx[a_idx] * prob
                    dy = action_dy[a_idx] * prob
                    ax_pol.annotate('', xy=(j + dx, y_pos + dy), xytext=(j, y_pos),
                                   arrowprops=dict(arrowstyle='->', color='steelblue',
                                                   lw=1 + 2 * prob))
    ax_pol.set_xticks([])
    ax_pol.set_yticks([])

    V_grid = V.reshape(sz, sz)
    im = ax_val.imshow(V_grid, cmap='RdYlGn')
    for r in range(sz):
        for c in range(sz):
            ax_val.text(c, r, f'{V_grid[r, c]:.1f}', ha='center', va='center',
                        fontsize=8, fontweight='bold')
    ax_val.set_title(val_title, fontsize=11, fontweight='bold')
    ax_val.set_xticks([])
    ax_val.set_yticks([])
    return im


mdp = GridWorldMDP(size=5, gamma=0.9)

np.random.seed(7)
old_policy = np.zeros((mdp.n_states, mdp.n_actions))
for s in range(mdp.n_states):
    a = np.random.randint(mdp.n_actions)
    old_policy[s, a] = 1.0

V_old, _, _ = policy_evaluation(mdp, old_policy)
new_policy = greedy_policy(mdp, V_old)
V_new, _, _ = policy_evaluation(mdp, new_policy)

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
draw_grid_policy_and_value(axes[0, 0], axes[0, 1], mdp, old_policy, V_old,
                           'Old (Random) Policy', 'V\u03c0_old')
im = draw_grid_policy_and_value(axes[1, 0], axes[1, 1], mdp, new_policy, V_new,
                                'Improved (Greedy) Policy', 'V\u03c0_new')
fig.colorbar(im, ax=axes, shrink=0.5, label='V(s)')
fig.suptitle('Policy Improvement: Before vs After', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('policy_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

improvement = V_new - V_old
print("\n\u2705 Policy Improvement Verification:")
print(f"   V_new >= V_old everywhere? {np.all(V_new >= V_old - 1e-10)}")
print(f"   Min improvement : {improvement.min():.4f}")
print(f"   Max improvement : {improvement.max():.4f}")
print(f"   Mean improvement: {improvement.mean():.4f}")

---

## 🔄 Section 4: Policy Iteration

**Policy iteration** alternates between evaluation and improvement until the policy stabilizes at the optimal policy $\pi^*$.

### Algorithm

1. Initialize $\pi$ arbitrarily
2. **Repeat:**
   - **(a) Evaluate:** Compute $V^\pi$ to convergence using iterative policy evaluation
   - **(b) Improve:** For every state, update the policy greedily:
     $$\pi_{\text{new}}(s) = \arg\max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s') \right]$$
   - **(c) Check:** If $\pi_{\text{new}} = \pi$, **stop** — the policy is optimal!
   - **(d)** $\pi \leftarrow \pi_{\text{new}}$

### Convergence

- Each improvement step produces a policy that is **strictly better** (or equal).
- For a finite MDP, the total number of deterministic policies is $|\mathcal{A}|^{|\mathcal{S}|}$.
- Since each step improves or stays the same, and there are finitely many policies, the algorithm must converge in a finite number of steps.
- In practice, convergence is typically very fast (far fewer iterations than the total number of policies).

In [ ]:
def policy_iteration(mdp, theta=1e-6):
    """Full policy iteration algorithm. Returns optimal V, policy, and history."""
    np.random.seed(0)
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    for s in range(mdp.n_states):
        a = np.random.randint(mdp.n_actions)
        policy[s, a] = 1.0

    history = []

    for outer_it in range(100):
        V, _, _ = policy_evaluation(mdp, policy, theta=theta)
        new_policy = greedy_policy(mdp, V)

        changed = []
        for s in range(mdp.n_states):
            if not np.allclose(policy[s], new_policy[s]):
                changed.append(s)

        history.append({
            'iteration': outer_it,
            'V': V.copy(),
            'policy': policy.copy(),
            'n_changed': len(changed),
            'changed_states': changed
        })

        if len(changed) == 0:
            print(f"   Policy iteration converged at outer iteration {outer_it}!")
            break

        policy = new_policy

    return V, policy, history


mdp = GridWorldMDP(size=5, gamma=0.9)
V_pi_star, pi_star, pi_history = policy_iteration(mdp)

n_show = min(len(pi_history), 6)
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
if n_show == 1:
    axes = axes.reshape(2, 1)

for idx in range(n_show):
    h = pi_history[idx]
    draw_grid_policy_and_value(
        axes[0, idx], axes[1, idx], mdp,
        h['policy'], h['V'],
        f"Policy (iter {h['iteration']})",
        f"V (iter {h['iteration']})\nchanged: {h['n_changed']}"
    )

fig.suptitle('Policy Iteration: Evolution of Policy and Value Function',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('policy_iteration_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

changes_per_iter = [h['n_changed'] for h in pi_history]
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.bar(range(len(changes_per_iter)), changes_per_iter, color='steelblue', edgecolor='navy')
ax2.set_xlabel('Outer Iteration', fontsize=12)
ax2.set_ylabel('Number of States with Policy Change', fontsize=12)
ax2.set_title('Policy Iteration Convergence: Policy Changes per Iteration',
              fontsize=13, fontweight='bold')
ax2.set_xticks(range(len(changes_per_iter)))
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('policy_iteration_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n\u2705 Policy iteration results:")
for h in pi_history:
    states_str = str(h['changed_states'][:10])
    if h['n_changed'] > 10:
        states_str += '...'
    print(f"   Iteration {h['iteration']}: {h['n_changed']} states changed {states_str}")

---

## ⚡ Section 5: Value Iteration

**Value iteration** combines the evaluation and improvement steps into a single update, making it more computationally efficient.

### Key Insight

Instead of fully evaluating a policy before improving, we can perform a single Bellman optimality backup:

### Algorithm

1. Initialize $V(s) = 0$ for all $s \in \mathcal{S}$
2. **Repeat:**
   - For each $s \in \mathcal{S}$:
     $$V(s) \leftarrow \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V(s') \right]$$
3. **Until** $\max_s |V_{\text{new}}(s) - V_{\text{old}}(s)| < \theta$

### Extracting the Optimal Policy

Once $V^*$ is found:

$$\pi^*(s) = \arg\max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^*(s') \right]$$

### Comparison with Policy Iteration

| Aspect | Policy Iteration | Value Iteration |
|--------|-----------------|----------------|
| Evaluation | Full (to convergence) | Truncated (1 sweep) |
| Improvement | Explicit greedy step | Implicit via max |
| Outer iterations | Few (policy space) | More (value space) |
| Per-iteration cost | High (full evaluation) | Low (single sweep) |
| Total work | Often similar | Often similar |

In [ ]:
def value_iteration(mdp, theta=1e-6, max_iter=1000, track_iters=None):
    """Value iteration algorithm. Returns optimal V, policy, deltas, and V snapshots."""
    V = np.zeros(mdp.n_states)
    deltas = []
    V_history = {0: V.copy()}
    if track_iters is None:
        track_iters = set()

    for it in range(1, max_iter + 1):
        delta = 0.0
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            v_old = V[s]
            q_values = np.array([
                mdp.R[s, a] + mdp.gamma * np.dot(mdp.P[s, a], V)
                for a in range(mdp.n_actions)
            ])
            V[s] = q_values.max()
            delta = max(delta, abs(v_old - V[s]))
        deltas.append(delta)
        if it in track_iters:
            V_history[it] = V.copy()
        if delta < theta:
            V_history[it] = V.copy()
            break

    optimal_policy = greedy_policy(mdp, V)
    return V, optimal_policy, deltas, V_history


mdp = GridWorldMDP(size=5, gamma=0.9)
track_at = {1, 5, 10, 20}
V_vi, pi_vi, vi_deltas, vi_hist = value_iteration(mdp, track_iters=track_at)

final_it = max(vi_hist.keys())
show_iters_vi = sorted(set([0, 1, 5, 10, 20, final_it]) & set(vi_hist.keys()))
show_iters_vi = show_iters_vi[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
vmin, vmax = V_vi.min(), V_vi.max()

for idx, (ax, it) in enumerate(zip(axes.flat, show_iters_vi)):
    V_grid = vi_hist[it].reshape(mdp.size, mdp.size)
    im = ax.imshow(V_grid, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    for r in range(mdp.size):
        for c in range(mdp.size):
            ax.text(c, r, f'{V_grid[r, c]:.1f}', ha='center', va='center',
                    fontsize=8, fontweight='bold')
    label = f'Iteration {it}' if it < final_it else f'Converged (iter {it})'
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

for ax in axes.flat[len(show_iters_vi):]:
    ax.axis('off')

fig.colorbar(im, ax=axes, shrink=0.6, label='V(s)')
fig.suptitle('Value Iteration: Value Function Evolution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('value_iteration_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

fig2, (ax_conv, ax_pol) = plt.subplots(1, 2, figsize=(14, 5))

ax_conv.semilogy(range(1, len(vi_deltas) + 1), vi_deltas, 'o-', color='crimson', markersize=3)
ax_conv.set_xlabel('Iteration', fontsize=12)
ax_conv.set_ylabel('Max |\u0394V|  (log scale)', fontsize=12)
ax_conv.set_title('Value Iteration Convergence', fontsize=13, fontweight='bold')
ax_conv.grid(True, alpha=0.3)

action_dx = {0: 0, 1: 0, 2: -0.35, 3: 0.35}
action_dy = {0: 0.35, 1: -0.35, 2: 0, 3: 0}
sz = mdp.size
ax_pol.set_xlim(-0.5, sz - 0.5)
ax_pol.set_ylim(-0.5, sz - 0.5)
ax_pol.set_aspect('equal')
for i in range(sz):
    for j in range(sz):
        ax_pol.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        fill=False, edgecolor='gray'))
for i in range(sz):
    for j in range(sz):
        s = i * sz + j
        y_pos = sz - 1 - i
        if s in mdp.terminal_states:
            ax_pol.plot(j, y_pos, 's', color='gold', markersize=18)
            ax_pol.text(j, y_pos, 'T', ha='center', va='center',
                       fontsize=9, fontweight='bold')
            continue
        for a_idx in range(mdp.n_actions):
            prob = pi_vi[s, a_idx]
            if prob > 0.01:
                dx = action_dx[a_idx] * prob
                dy = action_dy[a_idx] * prob
                ax_pol.annotate('', xy=(j + dx, y_pos + dy), xytext=(j, y_pos),
                               arrowprops=dict(arrowstyle='->', color='steelblue',
                                               lw=1 + 2 * prob))
ax_pol.set_title('Optimal Policy (from Value Iteration)', fontsize=13, fontweight='bold')
ax_pol.set_xticks([])
ax_pol.set_yticks([])

plt.tight_layout()
plt.savefig('value_iteration_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n\u2705 Value iteration converged in {len(vi_deltas)} iterations.")
print("\n   Comparing Policy Iteration vs Value Iteration results:")
print(f"   Max |V_PI - V_VI| = {np.max(np.abs(V_pi_star - V_vi)):.2e}")
print(f"   Results match: {np.allclose(V_pi_star, V_vi, atol=1e-5)}")

---

## 📜 Section 6: Convergence Guarantees

### Banach Fixed Point Theorem

The **Bellman optimality operator** $T$ is defined as:

$$(TV)(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V(s') \right]$$

**Theorem (Contraction).** $T$ is a $\gamma$-contraction in the $\ell_\infty$ (max) norm:

$$\|TV_1 - TV_2\|_\infty \leq \gamma \|V_1 - V_2\|_\infty$$

for any two value functions $V_1, V_2$.

**Consequences:**

1. There exists a **unique fixed point** $V^*$ such that $TV^* = V^*$.
2. Starting from any $V_0$, the iterates $V_k = T^k V_0$ converge to $V^*$.
3. **Rate of convergence:**

$$\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$$

4. **Iterations for $\epsilon$-convergence:**

$$k \geq \frac{\log\left(\|V_0 - V^*\|_\infty \,/\, \epsilon\right)}{\log(1/\gamma)}$$

In [ ]:
mdp = GridWorldMDP(size=5, gamma=0.9)
gamma = mdp.gamma

V1 = np.zeros(mdp.n_states)
V2 = np.random.uniform(-10, 10, mdp.n_states)
V2[mdp.terminal_states] = 0.0

max_iters = 60
diffs = [np.max(np.abs(V1 - V2))]
theoretical = [diffs[0] * gamma**k for k in range(max_iters + 1)]

for it in range(max_iters):
    for s in range(mdp.n_states):
        if s in mdp.terminal_states:
            continue
        q1 = [mdp.R[s, a] + gamma * np.dot(mdp.P[s, a], V1) for a in range(mdp.n_actions)]
        q2 = [mdp.R[s, a] + gamma * np.dot(mdp.P[s, a], V2) for a in range(mdp.n_actions)]
        V1[s] = max(q1)
        V2[s] = max(q2)
    diffs.append(np.max(np.abs(V1 - V2)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(range(len(diffs)), diffs, 'o-', color='steelblue', markersize=4,
            label=r'$\|V_1^k - V_2^k\|_\infty$ (empirical)', linewidth=2)
ax.semilogy(range(len(theoretical)), theoretical, '--', color='crimson',
            label=r'$\gamma^k \cdot \|V_1^0 - V_2^0\|_\infty$ (theoretical bound)',
            linewidth=2)
ax.set_xlabel('Iteration $k$', fontsize=12)
ax.set_ylabel(r'$\|V_1^k - V_2^k\|_\infty$ (log scale)', fontsize=12)
ax.set_title(f'Contraction Property of the Bellman Operator (\u03b3 = {gamma})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('contraction_property.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n\u2705 Contraction Property Demonstration:")
print(f"   Initial difference:  ||V1_0 - V2_0||_\u221e = {diffs[0]:.4f}")
print(f"   After 10 iterations: ||V1_10 - V2_10||_\u221e = {diffs[10]:.6f}")
print(f"   After 20 iterations: ||V1_20 - V2_20||_\u221e = {diffs[20]:.8f}")
print(f"   \u03b3 = {gamma}, so difference shrinks by factor {gamma} each iteration.")
print("   Both sequences converge to the SAME V* regardless of initialization.")

---

## 🔍 Section 7: Comparing Policy Iteration vs Value Iteration

| Feature | Policy Iteration | Value Iteration |
|---------|-----------------|----------------|
| **Inner loop** | Full policy evaluation (many sweeps) | Single Bellman backup |
| **Outer iterations** | Typically very few (2–5) | More iterations needed |
| **Per-iteration cost** | $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ per eval sweep | $O(|\mathcal{S}| \cdot |\mathcal{A}|)$ per sweep |
| **When to prefer** | Small state spaces | Large state spaces |
| **Guarantee** | Exact optimal policy | Converges to $V^*$ |

In [ ]:
import time

sizes = [4, 5, 6, 7, 8]
pi_times = []
vi_times = []
pi_iters_list = []
vi_iters_list = []
v_diffs = []

for sz in sizes:
    mdp_test = GridWorldMDP(size=sz, gamma=0.9)

    t0 = time.time()
    V_pi, pi_opt, pi_hist = policy_iteration(mdp_test, theta=1e-8)
    pi_time = time.time() - t0

    t0 = time.time()
    V_vi_test, _, vi_d, _ = value_iteration(mdp_test, theta=1e-8)
    vi_time = time.time() - t0

    pi_times.append(pi_time * 1000)
    vi_times.append(vi_time * 1000)
    pi_iters_list.append(len(pi_hist))
    vi_iters_list.append(len(vi_d))
    v_diffs.append(np.max(np.abs(V_pi - V_vi_test)))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.arange(len(sizes))
width = 0.35
axes[0].bar(x - width/2, pi_iters_list, width, label='Policy Iteration', color='steelblue')
axes[0].bar(x + width/2, vi_iters_list, width, label='Value Iteration', color='coral')
axes[0].set_xlabel('Grid Size', fontsize=12)
axes[0].set_ylabel('Iterations', fontsize=12)
axes[0].set_title('Iterations to Convergence', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'{s}\u00d7{s}' for s in sizes])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - width/2, pi_times, width, label='Policy Iteration', color='steelblue')
axes[1].bar(x + width/2, vi_times, width, label='Value Iteration', color='coral')
axes[1].set_xlabel('Grid Size', fontsize=12)
axes[1].set_ylabel('Time (ms)', fontsize=12)
axes[1].set_title('Computation Time', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s}\u00d7{s}' for s in sizes])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(x, v_diffs, color='seagreen', edgecolor='darkgreen')
axes[2].set_xlabel('Grid Size', fontsize=12)
axes[2].set_ylabel('Max |V_PI - V_VI|', fontsize=12)
axes[2].set_title('V* Agreement (should be ~0)', fontsize=13, fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels([f'{s}\u00d7{s}' for s in sizes])
axes[2].ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Policy Iteration vs Value Iteration: Head-to-Head Comparison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pi_vs_vi_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n\u2705 Comparison Summary:")
print(f"   {'Grid':<8} {'PI Iters':<10} {'VI Iters':<10} {'PI Time(ms)':<14} {'VI Time(ms)':<14} {'Max |V diff|'}")
print("   " + "-" * 70)
for i, sz in enumerate(sizes):
    print(f"   {sz}\u00d7{sz:<5} {pi_iters_list[i]:<10} {vi_iters_list[i]:<10} "
          f"{pi_times[i]:<14.2f} {vi_times[i]:<14.2f} {v_diffs[i]:.2e}")

---

## 🖼️ Section 8: Optimal Policy for Image Enhancement

Let us apply what we have learned to a practical **image enhancement MDP**.

### MDP Definition

| Component | Definition |
|-----------|------------|
| **States** | Discretized image quality levels: $\mathcal{S} = \{1, 2, \ldots, 10\}$ |
| **Actions** | $\mathcal{A} = \{\text{strong\_enhance},\; \text{mild\_enhance},\; \text{no\_change}\}$ |
| **Transitions** | `strong_enhance`: +3 quality (60%), +1 quality (40%) |
|  | `mild_enhance`: +1 quality (80%), +2 quality (20%) |
|  | `no_change`: stays at same quality (100%) |
| **Rewards** | +1 for each quality level increase, −0.1 per step (action cost), +10 bonus at quality 10 |
| **Terminal** | Quality level 10 |

**Goal:** Find the optimal strategy for enhancing images from any starting quality level.

In [ ]:
class ImageEnhancementMDP:
    """MDP for sequential image quality enhancement."""

    def __init__(self, n_quality=10, gamma=0.95):
        self.n_quality = n_quality
        self.n_states = n_quality
        self.n_actions = 3  # 0=strong_enhance, 1=mild_enhance, 2=no_change
        self.action_names = ['strong_enhance', 'mild_enhance', 'no_change']
        self.gamma = gamma
        self.terminal_states = [n_quality - 1]  # quality 10 (index 9)

        self.P = np.zeros((self.n_states, self.n_actions, self.n_states))
        self.R = np.zeros((self.n_states, self.n_actions))
        self._build()

    def _build(self):
        step_cost = -0.1
        terminal_bonus = 10.0

        for s in range(self.n_states):
            if s in self.terminal_states:
                for a in range(self.n_actions):
                    self.P[s, a, s] = 1.0
                    self.R[s, a] = 0.0
                continue

            s + 1  # 1-indexed quality

            # strong_enhance: +3 (60%), +1 (40%)
            s_big = min(s + 3, self.n_states - 1)
            s_small = min(s + 1, self.n_states - 1)
            self.P[s, 0, s_big] += 0.6
            self.P[s, 0, s_small] += 0.4
            expected_increase = 0.6 * min(3, self.n_states - 1 - s) + 0.4 * min(1, self.n_states - 1 - s)
            reward_0 = expected_increase + step_cost
            if s_big == self.n_states - 1:
                reward_0 += 0.6 * terminal_bonus
            if s_small == self.n_states - 1:
                reward_0 += 0.4 * terminal_bonus
            self.R[s, 0] = reward_0

            # mild_enhance: +1 (80%), +2 (20%)
            s_1 = min(s + 1, self.n_states - 1)
            s_2 = min(s + 2, self.n_states - 1)
            self.P[s, 1, s_1] += 0.8
            self.P[s, 1, s_2] += 0.2
            expected_increase_1 = 0.8 * min(1, self.n_states - 1 - s) + 0.2 * min(2, self.n_states - 1 - s)
            reward_1 = expected_increase_1 + step_cost
            if s_1 == self.n_states - 1:
                reward_1 += 0.8 * terminal_bonus
            if s_2 == self.n_states - 1:
                reward_1 += 0.2 * terminal_bonus
            self.R[s, 1] = reward_1

            # no_change: stay
            self.P[s, 2, s] = 1.0
            self.R[s, 2] = step_cost


img_mdp = ImageEnhancementMDP(n_quality=10, gamma=0.95)

def policy_iteration_general(mdp, theta=1e-8):
    """Policy iteration for a general MDP."""
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    for s in range(mdp.n_states):
        policy[s, 0] = 1.0

    for outer in range(100):
        V, _, _ = policy_evaluation(mdp, policy, theta=theta)
        new_policy = greedy_policy(mdp, V)
        if np.allclose(policy, new_policy):
            break
        policy = new_policy
    return V, policy

V_img, pi_img = policy_iteration_general(img_mdp)

optimal_actions = np.argmax(pi_img, axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {0: '#e74c3c', 1: '#3498db', 2: '#95a5a6'}
labels_map = {0: 'Strong Enhance', 1: 'Mild Enhance', 2: 'No Change'}
bar_colors = [colors_map[a] for a in optimal_actions]
bars = ax1.bar(range(1, 11), [1] * 10, color=bar_colors, edgecolor='black', linewidth=1.2)
ax1.set_xlabel('Image Quality Level', fontsize=12)
ax1.set_ylabel('')
ax1.set_title('Optimal Action per Quality State', fontsize=13, fontweight='bold')
ax1.set_xticks(range(1, 11))
ax1.set_yticks([])
for i, a in enumerate(optimal_actions):
    ax1.text(i + 1, 0.5, labels_map[a], ha='center', va='center',
             fontsize=8, fontweight='bold', rotation=90, color='white')
patches = [mpatches.Patch(color=colors_map[k], label=labels_map[k]) for k in [0, 1, 2]]
ax1.legend(handles=patches, loc='upper right', fontsize=10)

ax2.plot(range(1, 11), V_img, 'o-', color='steelblue', linewidth=2.5, markersize=8)
ax2.set_xlabel('Image Quality Level', fontsize=12)
ax2.set_ylabel('V*(s)', fontsize=12)
ax2.set_title('Optimal Value Function V*', fontsize=13, fontweight='bold')
ax2.set_xticks(range(1, 11))
ax2.grid(True, alpha=0.3)
for i, v in enumerate(V_img):
    ax2.annotate(f'{v:.1f}', (i + 1, v), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=9)

plt.suptitle('Image Enhancement MDP: Optimal Policy',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('image_enhancement_policy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n\u2705 Optimal Image Enhancement Policy:")
for s in range(img_mdp.n_states):
    action = labels_map[optimal_actions[s]]
    print(f"   Quality {s+1:2d}: {action:<18s} (V* = {V_img[s]:.2f})")

print("\n\ud83d\udca1 Interpretation:")
print("   \u2022 At low quality levels, strong enhancement is optimal (high expected gain).")
print("   \u2022 At higher quality levels, mild enhancement becomes safer and preferred.")
print("   \u2022 The 'no_change' action is never optimal because the terminal bonus incentivizes progress.")

In [ ]:
def simulate_policy(mdp, policy, start_state, max_steps=50):
    """Simulate one episode following the given policy."""
    s = start_state
    total_reward = 0.0
    steps = 0
    for t in range(max_steps):
        if s in mdp.terminal_states:
            break
        a = np.random.choice(mdp.n_actions, p=policy[s])
        total_reward += (mdp.gamma ** t) * mdp.R[s, a]
        s_next = np.random.choice(mdp.n_states, p=mdp.P[s, a])
        s = s_next
        steps += 1
    reached_goal = (s in mdp.terminal_states)
    return steps, total_reward, reached_goal


np.random.seed(42)
n_episodes = 100
start_state = 0  # quality level 1

random_policy = np.ones((img_mdp.n_states, img_mdp.n_actions)) / img_mdp.n_actions

opt_steps, opt_rewards, opt_goals = [], [], []
rand_steps, rand_rewards, rand_goals = [], [], []

for ep in range(n_episodes):
    s, r, g = simulate_policy(img_mdp, pi_img, start_state)
    opt_steps.append(s)
    opt_rewards.append(r)
    opt_goals.append(g)

    s, r, g = simulate_policy(img_mdp, random_policy, start_state, max_steps=200)
    rand_steps.append(s)
    rand_rewards.append(r)
    rand_goals.append(g)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(opt_steps, bins=15, alpha=0.7, color='steelblue', label='Optimal', edgecolor='navy')
axes[0].hist(rand_steps, bins=15, alpha=0.5, color='coral', label='Random', edgecolor='darkred')
axes[0].set_xlabel('Steps to Quality 10', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Steps to Reach Max Quality', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(opt_rewards, bins=15, alpha=0.7, color='steelblue', label='Optimal', edgecolor='navy')
axes[1].hist(rand_rewards, bins=15, alpha=0.5, color='coral', label='Random', edgecolor='darkred')
axes[1].set_xlabel('Total Discounted Reward', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Total Reward Distribution', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

categories = ['Optimal\nPolicy', 'Random\nPolicy']
avg_steps = [np.mean(opt_steps), np.mean(rand_steps)]
avg_rewards = [np.mean(opt_rewards), np.mean(rand_rewards)]
x_pos = np.arange(2)

bars = axes[2].bar(x_pos - 0.2, avg_steps, 0.35, label='Avg Steps', color='steelblue')
ax2_twin = axes[2].twinx()
bars2 = ax2_twin.bar(x_pos + 0.2, avg_rewards, 0.35, label='Avg Reward', color='seagreen')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(categories)
axes[2].set_ylabel('Avg Steps', fontsize=12, color='steelblue')
ax2_twin.set_ylabel('Avg Reward', fontsize=12, color='seagreen')
axes[2].set_title('Policy Comparison Summary', fontsize=13, fontweight='bold')
lines = [bars, bars2]
labels = ['Avg Steps', 'Avg Reward']
axes[2].legend(lines, labels, loc='upper center', fontsize=10)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle(f'Optimal vs Random Policy: {n_episodes} Episodes from Quality 1',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('policy_simulation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n\u2705 Simulation Results ({n_episodes} episodes, starting at quality 1):")
print(f"\n   {'Metric':<25} {'Optimal Policy':<18} {'Random Policy':<18}")
print("   " + "-" * 60)
print(f"   {'Avg steps to quality 10':<25} {np.mean(opt_steps):<18.2f} {np.mean(rand_steps):<18.2f}")
print(f"   {'Avg total reward':<25} {np.mean(opt_rewards):<18.2f} {np.mean(rand_rewards):<18.2f}")
print(f"   {'Success rate':<25} {np.mean(opt_goals)*100:<17.1f}% {np.mean(rand_goals)*100:<17.1f}%")
print(f"   {'Std steps':<25} {np.std(opt_steps):<18.2f} {np.std(rand_steps):<18.2f}")
print("\n\ud83d\udca1 The optimal policy reaches maximum quality significantly faster and")
print("   accumulates much higher reward than a random exploration strategy.")

---

## 🎯 Summary

### Algorithm Comparison

| Algorithm | Input | Output | Key Update Rule | Convergence |
|-----------|-------|--------|-----------------|-------------|
| **Policy Evaluation** | MDP + $\pi$ | $V^\pi$ | $V(s) \leftarrow \sum_a \pi(a|s)[R + \gamma \sum_{s'} P \cdot V(s')]$ | $\gamma$-contraction |
| **Policy Improvement** | $V^\pi$ | $\pi' \geq \pi$ | $\pi'(s) = \arg\max_a Q^\pi(s,a)$ | Monotonic |
| **Policy Iteration** | MDP | $\pi^*, V^*$ | Alternate eval + improve | Finite steps |
| **Value Iteration** | MDP | $V^*, \pi^*$ | $V(s) \leftarrow \max_a [R + \gamma \sum_{s'} P \cdot V(s')]$ | $\gamma$-contraction |

### Key Convergence Results

- The Bellman operator is a **$\gamma$-contraction** in the max norm
- Both policy iteration and value iteration converge to the **unique optimal** $V^*$ and $\pi^*$
- Error decreases **exponentially** at rate $\gamma^k$
- Policy iteration converges in at most $|\mathcal{A}|^{|\mathcal{S}|}$ steps (but usually far fewer)

### Connection to Image Processing

- Image enhancement can be modeled as an MDP with quality states and enhancement actions
- Optimal policies prescribe **aggressive enhancement** at low quality and **cautious refinement** at high quality
- These dynamic programming methods provide the **theoretical foundation** for more advanced deep RL approaches to image processing

---

### ➡️ Next: Module 4.1 — Introduction to Deep Learning for RL

---

*\u00a9 FlashVision — Reinforcement Learning for Image Processing*